# Chapter 10 companion: closed-loop time response and steady-state accuracy.

*Computers in the Loop: Digital Control from Sampling to Networked Agents*: EECE 5610 lecture notes prepared and revised by Milad Siami.

Chapters 1–11 of the notes are based on the lecture notes of Prof. Mario Sznaier; all credits are in `ATTRIBUTION.md`.

Loop: R -> (+/-) -> sampler -> D(z) -> ZOH -> G_p(s) -> C, unity feedback.

Reproduces:
  * Example 1, G_p = 4/(s+2), T = 0.1 s: G(z) = 0.3625/(z - 0.8187), closed-loop pole
    0.4562, tau = 0.127 s, c(kT) -> 2/3; the pole 3e^{-2T} - 2 turns negative for
    T > 0.203 s and leaves the unit circle for T > 0.549 s;
  * Example 2, G_p = 1/(s(s+1)), T = 1 s: poles 0.5 +- j0.618, zeta = 0.25,
    wn = 0.92 rad/s, tau = 4.36 s; peak 1.40 at the samples and 1.45 between them;
  * the effect of T on the same loop (table), and the limit T = 3.92 s;
  * Example B: an accumulator D = z/(z-1) removes the 50% step error;
  * the laser loop G_p = 2/((s+1)(s+4)) with D = k: stable for k < 54.7 (T = 0.1 s)
    and k < 6.78 (T = 1 s);
  * the exit ticket: Example 2 with K = 2.

Run the cells from top to bottom (in Colab: *Runtime > Run all*). Figures are also saved to a `figures` folder.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy import signal
from scipy.linalg import expm
from scipy.optimize import brentq

FIG = Path.cwd() / "figures"
FIG.mkdir(exist_ok=True)

In [ ]:
def zoh(num, den, T):
    """ZOH equivalent, numerator padded to the length of the denominator."""
    numz, denz, _ = signal.cont2discrete((num, den), T, method="zoh")
    numz = np.atleast_2d(numz)[0]
    return np.r_[np.zeros(len(denz) - len(numz)), numz], denz

In [ ]:
def closed_loop(numz, denz, K=1.0):
    """C/R = K G/(1 + K G) for G = numz/denz (same lengths)."""
    return K * numz, denz + K * numz

In [ ]:
def specs(zp, T):
    """zeta, wn, tau from a z-plane pole r e^{j theta}."""
    r, th = abs(zp), abs(np.angle(zp))
    L = np.hypot(np.log(r), th)
    return -np.log(r) / L, L / T, -T / np.log(r)

In [ ]:
def hybrid_step(num, den, T, D=1.0, n=20, sub=50):
    """Unit-step response of the sampled loop with D = constant gain, between the
    samples too: the plant is integrated exactly on a grid of T/sub."""
    A, B, C, Dd = signal.tf2ss(num, den)
    nx = A.shape[0]
    M = np.zeros((nx + 1, nx + 1))
    M[:nx, :nx], M[:nx, nx:] = A, B
    E = expm(M * T / sub)
    Ph, Ga = E[:nx, :nx], E[:nx, nx:]
    x = np.zeros((nx, 1))
    t, y = [], []
    for k in range(n):
        u = D * (1.0 - (C @ x).item())             # sample the error, hold the control
        for j in range(sub):
            t.append((k + j / sub) * T)
            y.append((C @ x).item())
            x = Ph @ x + Ga * u
    return np.array(t), np.array(y)

In [ ]:
# ---------------------------------------------------------------- Example 1
T = 0.1

In [ ]:
nz, dz = zoh([4], [1, 2], T)
nc, dc = closed_loop(nz, dz)
pole = -dc[1]
zeta, wn, tau = specs(pole, T)
c = signal.lfilter(nc, dc, np.ones(6))
print("Example 1: G_p = 4/(s+2), T = 0.1 s")
print(f"  G(z) = {nz[-1]:.4f}/(z - {-dz[1]:.4f});  C/R = {nc[-1]:.4f}/(z - {pole:.4f})")
print(f"  tau = -T/ln r = {tau:.3f} s (analog loop 4/(s+6): {1 / 6:.3f} s);  equivalent s-pole {np.log(pole) / T:.3f}")
print("  c(kT) =", np.round(c, 4), " -> 2/3")
print(f"  closed-loop pole 3e^(-2T) - 2 < 0 for T > ln(1.5)/2 = {np.log(1.5) / 2:.3f} s;"
      f" < -1 for T > ln(3)/2 = {np.log(3) / 2:.3f} s")

In [ ]:
# ---------------------------------------------------------------- Example 2
T = 1.0

In [ ]:
nz, dz = zoh([1], [1, 1, 0], T)
nc, dc = closed_loop(nz, dz)
zp = np.roots(dc)
zp = zp[np.argmax(zp.imag)]
zeta, wn, tau = specs(zp, T)
c = signal.lfilter(nc, dc, np.ones(21))
t, y = hybrid_step([1], [1, 1, 0], T, n=21)
print("\nExample 2: G_p = 1/(s(s+1)), T = 1 s")
print(f"  G(z) = ({nz[1]:.4f} z + {nz[2]:.4f})/(z^2 - {-dz[1]:.4f} z + {dz[2]:.4f})")
print(f"  characteristic polynomial z^2 - {-dc[1]:.4f} z + {dc[2]:.4f}: poles {zp.real:.3f} +- j{zp.imag:.3f}"
      f" = {abs(zp):.3f} at +-{np.angle(zp):.3f} rad")
print(f"  zeta = {zeta:.3f}, wn = {wn:.3f} rad/s, tau = {tau:.2f} s, Mp = {100 * np.exp(-np.pi * zeta / np.sqrt(1 - zeta**2)):.1f}%,"
      f" {2 * np.pi / np.angle(zp):.1f} samples per cycle")
print("  c(k), k = 0..10:", np.round(c[:11], 3))
print(f"  peak at the samples {c.max():.3f}; true peak between samples {y.max():.3f} at t = {t[y.argmax()]:.2f} s")
k_in = np.nonzero(np.abs(c - 1) > 0.02)[0][-1] + 1
print(f"  samples within 2% from k = {k_in} on")
_, ya = signal.step(([1], [1, 1, 1]), T=np.linspace(0, 20, 20001))
print(f"  analog loop 1/(s^2+s+1): overshoot {100 * (ya.max() - 1):.0f}%")

In [ ]:
# ---------------------------------------------------------------- effect of T
print("\nEffect of T on Example 2 (z^2 + (T-2) z + (1 - T e^-T) = 0)")

In [ ]:
print("   T     poles                    |z|     zeta   peak of samples")

In [ ]:
for T in (0.1, 0.5, 1.0, 2.0, 4.0):
    nz, dz = zoh([1], [1, 1, 0], T)
    nc, dc = closed_loop(nz, dz)
    rts = np.roots(dc)
    rmax = np.max(np.abs(rts))
    cs = signal.lfilter(nc, dc, np.ones(int(60 / T) + 1))
    if rmax < 1 and np.all(np.abs(rts.imag) > 1e-9):
        zp = rts[np.argmax(rts.imag)]
        zstr = f"{specs(zp, T)[0]:.2f}"
        pstr = f"{np.round(zp.real, 3) + 0.0:.3f} +- j{zp.imag:.3f}"
    else:
        zstr, pstr = " --", ", ".join(f"{r.real:.3f}" for r in rts)
    print(f"  {T:3.1f}   {pstr:22s}  {rmax:.3f}   {zstr:5s}  {cs.max() if rmax < 1 else float('inf'):.2f}")

In [ ]:
Tlim = brentq(lambda T: T * (1 + np.exp(-T)) - 4, 1, 6)
print(f"  a real pole reaches z = -1 when T (1 + e^-T) = 4: T = {Tlim:.2f} s")

In [ ]:
# ---------------------------------------------------------------- Example B: digital integrator
T = np.log(2)

In [ ]:
G = (np.array([0, 0.5]), np.array([1, -0.5]))
print("\nExample B: ZOH + 1/(s+1), T = ln 2, G(z) = 0.5/(z - 0.5)")
print("  without D: c(k) =", np.round(signal.lfilter([0, 0.5], [1, 0], np.ones(6)), 4), " (C/R = 0.5/z)")
nc, dc = [0, 0.5, 0], [1, -1, 0.5]                 # D G/(1 + D G) with D = z/(z-1)
c = signal.lfilter(nc, dc, np.ones(12))
zp = np.roots(dc)[0]
print("  with D = z/(z-1): c(k) =", np.round(c[:9], 4))
print(f"  poles 0.5 +- j0.5: r = {abs(zp):.3f}, theta = pi/4, zeta = {specs(zp, T)[0]:.2f}")
ramp = np.arange(400) * T
e = ramp - signal.lfilter(nc, dc, ramp)
print(f"  ramp error with D: {e[-1]:.4f} = T = ln 2")

In [ ]:
# ---------------------------------------------------------------- the laser loop with D = k
print("\nLaser loop, G_p = 2/((s+1)(s+4)), D = k")

In [ ]:
for T in (0.1, 1.0):
    nz, dz = zoh([2], [1, 5, 4], T)
    f = lambda kk: np.max(np.abs(np.roots(dz + kk * nz))) - 1
    kmax = brentq(f, 0.5, 5000 if T < 0.5 else 50)
    print(f"  T = {T}: G(z) = ({nz[1]:.4f} z + {nz[2]:.4f})/(z^2 - {-dz[1]:.4f} z + {dz[2]:.4f}),"
          f" stable for -2 < k < {kmax:.2f};  K_p = k G(1) = k x {np.sum(nz) / np.sum(dz):.3f}")

In [ ]:
print(f"  5% step error needs k >= {2 / 0.05 - 2:.0f}; with T = 1 s the best is 2/(2 + 6.78) = {2 / 8.78:.2f}")

In [ ]:
# ---------------------------------------------------------------- exit ticket
T = 1.0

In [ ]:
nz, dz = zoh([1], [1, 1, 0], T)
nc, dc = closed_loop(nz, dz, K=2.0)
zp = np.roots(dc)
zp = zp[np.argmax(zp.imag)]
c = signal.lfilter(nc, dc, np.ones(60))
print(f"\nExit ticket, K = 2: z^2 - {-dc[1]:.3f} z + {dc[2]:.3f}, poles {zp.real:.3f} +- j{zp.imag:.3f},"
      f" |z| = {abs(zp):.3f}, zeta = {specs(zp, T)[0]:.2f}, peak of the samples {c.max():.2f}, ramp error 1/K_v = 0.5")

In [ ]:
# ---------------------------------------------------------------- figure
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))

In [ ]:
T = 1.0
t, y = hybrid_step([1], [1, 1, 0], T, n=21)
nz, dz = zoh([1], [1, 1, 0], T)
c = signal.lfilter(*closed_loop(nz, dz), np.ones(21))
ta = np.linspace(0, 20, 400)
_, ya = signal.step(([1], [1, 1, 1]), T=ta)
axes[0].plot(ta, ya, "k--", lw=1, label="analog loop")
axes[0].plot(t, y, color="gray", label="sampled loop c(t)")
axes[0].plot(np.arange(21), c, "o", color="C3", ms=4, label="samples c(kT)")
axes[0].set(xlabel="t (s)", title="Example 2, T = 1 s")
axes[0].legend(fontsize=8)

In [ ]:
for T in (0.1, 0.5, 1.0, 2.0):
    nz, dz = zoh([1], [1, 1, 0], T)
    n = int(25 / T) + 1
    axes[1].plot(np.arange(n) * T, signal.lfilter(*closed_loop(nz, dz), np.ones(n)), ".-", ms=3, label=f"T = {T} s")

In [ ]:
axes[1].set(xlabel="t (s)", title="Example 2: the samples for several T")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "ch10_time_response.png", dpi=150)
print("\nSaved figures/ch10_time_response.png")

## Explore




In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
r"""**The running loop: gain and sampling period.** A zero-order hold drives
$G_p(s) = 1/(s(s+1))$, with $D(z) = K$ and unity feedback. Start at $K = 1$, $T = 1$ s: the
poles sit at $0.5 \pm j0.618$ and the samples peak at 1.40, while the output between the samples
peaks at 1.45. Raise $K$ toward 2.39, or $T$ toward 3.92 s, and watch the poles reach the unit
circle and the response ring, then grow. Small $T$ brings the samples back to the analog loop.
"""
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"
T_END = 20.0


def _analog_step(K, t):
    """Step response of K/(s^2 + s + K) (the loop without the sampler)."""
    if abs(K - 0.25) < 1e-9:                           # double root: nudge it apart
        K += 1e-7
    s1, s2 = np.roots([1.0, 1.0, K]).astype(complex)
    return np.real(1 + (s2 * np.exp(s1 * t) - s1 * np.exp(s2 * t)) / (s1 - s2))


def explore(K=1.0, T=1.0):
    p = np.exp(-T)
    b1, b0 = T - 1 + p, 1 - p - T * p                  # G(z) = (b1 z + b0)/((z - 1)(z - p))
    poles = np.roots([1.0, K * b1 - (1 + p), p + K * b0])
    rmax = np.max(np.abs(poles))
    stable = rmax < 1 - 1e-12

    # the loop, exact between the samples: states c and dc/dt, input m held over each period
    n = int(np.ceil(T_END / T))
    tau = np.linspace(0, T, 21)[:-1]
    x1 = x2 = 0.0
    ck, tf, cf = [], [], []
    for k in range(n + 1):
        ck.append(x1)
        m = K * (1.0 - x1)
        tf.append(k * T + tau)
        cf.append(x1 + x2 * (1 - np.exp(-tau)) + m * (tau - 1 + np.exp(-tau)))
        x1, x2 = x1 + x2 * (1 - p) + m * b1, x2 * p + m * (1 - p)
    ck, tf, cf = np.array(ck), np.concatenate(tf), np.concatenate(cf)

    t = np.linspace(0, T_END, 800)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.0), gridspec_kw={"width_ratios": [2.1, 1]})
    ax1.plot(t, _analog_step(K, t), "--", color=GRAY, lw=1.4, label="analog loop")
    ax1.plot(tf, cf, color=ORANGE, lw=1.5, label="c(t), sampled loop")
    ax1.plot(np.arange(n + 1) * T, ck, "o", color=ORANGE, ms=5 if T >= 0.3 else 2.5,
             label="samples c(kT)")
    ax1.axhline(1, color="k", lw=0.6)
    ax1.set_xlim(0, T_END)
    if stable:
        ax1.set_ylim(min(-0.2, cf.min() - 0.1), max(2.2, cf.max() + 0.1))
    else:
        ax1.set_ylim(-2.0, 4.0)
    ax1.set_xlabel("t (s)")
    ax1.set_ylabel("c")
    ax1.set_title(f"unit step, K = {K:g}, T = {T:g} s" + ("" if stable else ": unstable"), fontsize=11)
    ax1.legend(fontsize=10, loc="lower right")

    th = np.linspace(0, 2 * np.pi, 300)
    ax2.plot(np.cos(th), np.sin(th), color=GRAY, lw=1)
    ax2.plot(poles.real, poles.imag, "x", color=ORANGE, ms=11, mew=2.5)
    ax2.axhline(0, color="k", lw=0.5)
    ax2.axvline(0, color="k", lw=0.5)
    lim = max(1.3, 1.1 * rmax)
    ax2.set_xlim(-lim, lim)
    ax2.set_ylim(-lim, lim)
    ax2.set_aspect("equal")
    ax2.set_xlabel("Re z")
    ax2.set_ylabel("Im z")
    ax2.set_title("closed-loop poles", fontsize=11)
    fig.tight_layout()

    print(f"G(z) = ({b1:.4f} z + {b0:.4f}) / ((z - 1)(z - {p:.4f}));  "
          f"1 + KG = 0: z^2 {K * b1 - (1 + p):+.4f} z {p + K * b0:+.4f}")
    print("closed-loop poles:", ", ".join(f"{z.real:.3f}{z.imag:+.3f}j" for z in poles),
          f"   max |z| = {rmax:.3f}")
    if stable:
        line = f"stable: samples peak at {ck.max():.2f}, c(t) peaks at {cf.max():.2f}"
        zp = poles[np.argmax(poles.imag)]
        if abs(zp.imag) > 1e-9:
            s = np.log(zp) / T
            line += f";  zeta = {-s.real / abs(s):.2f}, wn = {abs(s):.2f} rad/s"
        print(line)
    else:
        print("not stable: a pole is on or outside the unit circle")
    plt.show()


try:
    from ipywidgets import interact, FloatSlider
    interact(explore,
             K=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.05, description="K",
                           continuous_update=False),
             T=FloatSlider(value=1.0, min=0.1, max=4.0, step=0.05, description="T (s)",
                           continuous_update=False))
except ImportError:
    explore()

## Hands-on

**Hands-on, Chapter 10.** These cells go with the Hands-on exercises at the end of
Chapter 10. Do each exercise on paper first and write down your prediction; then use the
matching section below to check it. Each section has sliders: change one thing at a time.


In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"

In [ ]:
def read_pole(z, T):
    """zeta, wn, tau of a z-plane pole, angle taken in the primary strip."""
    r, th = abs(z), abs(np.angle(z))
    L = np.hypot(np.log(r), th)
    return -np.log(r) / L, L / T, -T / np.log(r)

In [ ]:
def proto_step(zeta, wn, t):
    """Unit step response of wn^2 / (s^2 + 2 zeta wn s + wn^2), 0 < zeta < 1."""
    wd = wn * np.sqrt(1 - zeta ** 2)
    return 1 - np.exp(-zeta * wn * t) * (np.cos(wd * t) + zeta * wn / wd * np.sin(wd * t))

In [ ]:
# ---------------------------------------------------------------- H1
def h1_two_drones(T_A=0.005, T_B=0.05):
    """H1: loop A has poles 0.9 +- j0.1, loop B has 0.6 +- j0.45; time axis in seconds."""
    fig, ax = plt.subplots(figsize=(10, 3.8))
    t = np.linspace(0, 1.2, 2000)
    for z, T, col, name in [(0.9 + 0.1j, T_A, BLUE, "A"), (0.6 + 0.45j, T_B, ORANGE, "B")]:
        zeta, wn, tau = read_pole(z, T)
        Mp = np.exp(-np.pi * zeta / np.sqrt(1 - zeta ** 2))
        ax.plot(t, proto_step(zeta, wn, t), color=col, lw=1.4,
                label=f"{name}: T = {1000 * T:g} ms, zeta {zeta:.2f}, tau {tau:.3f} s, "
                      f"Ts {4 * tau:.2f} s, Mp {100 * Mp:.0f}%")
        k = np.arange(0, int(1.2 / T) + 1)
        ax.plot(k * T, proto_step(zeta, wn, k * T), "o", color=col, ms=3)
    ax.axhline(1, color=GRAY, lw=0.6)
    ax.set_xlabel("t (s)")
    ax.set_ylim(0, 1.4)
    ax.set_title("same z-plane poles, different sampling periods", fontsize=11)
    ax.legend(loc="lower right", fontsize=9)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H2
def h2_gimbal_mode(T=0.02):
    """H2: the mode r^k cos(k theta) of the pole -0.4 + j0.3, true angle vs misread angle."""
    z = -0.4 + 0.3j
    r = abs(z)
    th_true, th_wrong = np.angle(z), np.arctan(0.3 / -0.4)
    k = np.arange(0, 16)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.8), gridspec_kw={"width_ratios": [2, 1]})
    ax1.plot(k * T, r ** k * np.cos(k * th_true), "o-", color=ORANGE, ms=5, lw=1,
             label=f"true angle {th_true:.3f} rad")
    ax1.plot(k * T, r ** k * np.cos(k * abs(th_wrong)), "s--", color=TEAL, ms=4, lw=1,
             label=f"misread angle {abs(th_wrong):.3f} rad")
    ax1.axhline(0, color=GRAY, lw=0.6)
    ax1.set_xlabel("t (s)")
    zeta, wn, tau = read_pole(z, T)
    ax1.set_title(f"T = {1000 * T:g} ms: zeta {zeta:.2f}, wn {wn:.0f} rad/s, tau {1000 * tau:.0f} ms",
                  fontsize=11)
    ax1.legend(loc="upper right", fontsize=9)
    th = np.linspace(0, 2 * np.pi, 300)
    ax2.plot(np.cos(th), np.sin(th), color=GRAY, lw=1)
    ax2.plot([z.real, z.real], [z.imag, -z.imag], "x", color=ORANGE, ms=12, mew=2.5)
    ax2.plot([-z.real, -z.real], [z.imag, -z.imag], "x", color=TEAL, ms=9, mew=1.5)
    ax2.set_aspect("equal")
    ax2.set_xlim(-1.3, 1.3)
    ax2.set_ylim(-1.2, 1.2)
    ax2.set_title("true poles (orange), misread (teal)", fontsize=10)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H3
def h3_tower_sway(T=0.3):
    """H3: the tower mode s = -0.5 +- j6 sampled every T; what the samples suggest."""
    s = -0.5 + 6j
    z = np.exp(s * T)
    zeta, wn, tau = read_pole(z, T)
    t = np.linspace(0, 12, 3000)
    k = np.arange(0, int(12 / T) + 1)
    th = np.angle(z)                              # primary-strip angle
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.8), gridspec_kw={"width_ratios": [2, 1]})
    ax1.plot(t, np.exp(-0.5 * t) * np.cos(6 * t), color=GRAY, lw=1, label="tower sway (true)")
    ax1.plot(t, np.exp(-0.5 * t) * np.cos(th / T * t), "--", color=BLUE, lw=1.4,
             label="mode the samples suggest")
    ax1.plot(k * T, np.exp(-0.5 * k * T) * np.cos(6 * k * T), "o", color=ORANGE, ms=5, label="samples")
    ax1.set_xlabel("t (s)")
    ax1.set_title(f"T = {T:.3f} s: angle {th:.3f} rad, apparent zeta {zeta:.2f}, "
                  f"wn {wn:.2f} rad/s (true 0.08, 6.02)", fontsize=11)
    ax1.legend(loc="upper right", fontsize=9)
    c = np.linspace(0, 2 * np.pi, 300)
    ax2.plot(np.cos(c), np.sin(c), color=GRAY, lw=1)
    ax2.plot([z.real, z.real], [z.imag, -z.imag], "x", color=ORANGE, ms=12, mew=2.5)
    ax2.set_aspect("equal")
    ax2.set_xlim(-1.3, 1.3)
    ax2.set_ylim(-1.2, 1.2)
    ax2.set_title("pole of the sampled mode", fontsize=10)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H4
def h4_conveyor(K=0.55):
    """H4: belt speed G(z) = 1.5/(z - 0.25), T = 0.1 s, accumulator u_k = u_{k-1} + K e_k, ramp 2 m/s^2."""
    T, p, b, rate = 0.1, 0.25, 1.5, 2.0
    n = 40
    r = rate * T * np.arange(n + 1)
    y = np.zeros(n + 1)
    u = np.zeros(n + 1)
    for k in range(n):
        u[k] = (u[k - 1] if k >= 1 else 0.0) + K * (r[k] - y[k])
        y[k + 1] = p * y[k] + b * u[k]
    e = r - y
    poles = np.roots([1, -(1.25 - 1.5 * K), 0.25])
    if max(abs(poles)) >= 1:
        verdict, zeta = "unstable", float("nan")
    else:
        zp = poles[np.argmax(poles.imag)]
        zeta = read_pole(zp, T)[0]
        ok = zeta >= 0.5 and 0.1 / K <= 0.2
        verdict = "PASS" if ok else "FAIL"
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.8), gridspec_kw={"width_ratios": [2, 1]})
    ax1.plot(T * np.arange(n + 1), e, "o-", color=ORANGE, ms=4, lw=1, label="following error (m/s)")
    ax1.axhline(0.2, color=TEAL, ls="--", lw=1, label="spec 0.2 m/s")
    ax1.axhline(0.1 / K, color=BLUE, lw=0.8, label=f"steady error {0.1 / K:.3f} m/s")
    ax1.set_ylim(-0.1, 0.6)
    ax1.set_xlabel("t (s)")
    ax1.set_title(f"K = {K:.3f}: zeta {zeta:.2f}, |z| {max(abs(poles)):.3f}: {verdict}", fontsize=11)
    ax1.legend(loc="upper right", fontsize=9)
    c = np.linspace(0, 2 * np.pi, 300)
    ax2.plot(np.cos(c), np.sin(c), color=GRAY, lw=1)
    ax2.plot(0.5 * np.cos(c), 0.5 * np.sin(c), ":", color=GRAY, lw=1)
    ax2.plot(poles.real, poles.imag, "x", color=ORANGE, ms=12, mew=2.5)
    ax2.set_aspect("equal")
    ax2.set_xlim(-1.3, 1.3)
    ax2.set_ylim(-1.2, 1.2)
    ax2.set_title("closed-loop poles", fontsize=10)
    plt.show()

In [ ]:
try:
    from ipywidgets import interact, FloatSlider
    interact(h1_two_drones,
             T_A=FloatSlider(value=0.005, min=0.002, max=0.05, step=0.001, readout_format=".3f",
                             description="T_A (s)", continuous_update=False),
             T_B=FloatSlider(value=0.05, min=0.01, max=0.2, step=0.005, readout_format=".3f",
                             description="T_B (s)", continuous_update=False))
    interact(h2_gimbal_mode, T=FloatSlider(value=0.02, min=0.005, max=0.1, step=0.005,
                                           readout_format=".3f", description="T (s)",
                                           continuous_update=False))
    interact(h3_tower_sway, T=FloatSlider(value=0.3, min=0.05, max=1.2, step=0.001,
                                          readout_format=".3f", description="T (s)",
                                          continuous_update=False))
    interact(h4_conveyor, K=FloatSlider(value=0.55, min=0.05, max=1.7, step=0.01,
                                        readout_format=".2f", description="K",
                                        continuous_update=False))
except ImportError:
    h1_two_drones()
    h2_gimbal_mode()
    h3_tower_sway()
    h4_conveyor()